In [ ]:
# Rebin bispectrum data from dk_ori -> dk = N * dk_ori

In [ ]:
import numpy as np
import math

In [ ]:
dk_ori = 0.005

In [ ]:
# Triangle configurations

dk = 3 # to rebin data to 3*dk
cf = 2 # center of the first bin in terms of dk_ori
Nb = 5 # number of bins
i_open = 1 #1 for including open triangles, 0 for no
############################

NBmax = (Nb-0.5)*dk + cf

k1C_r, k2C_r, k3C_r = [], [], []
GetIdxB = np.zeros([Nb, Nb, Nb])

Ntri_tot = 0
for i in np.arange(cf, NBmax+0.5, dk):
    for j in np.arange(cf, i+1, dk):
        for l in np.arange(cf, j+1, dk):
            if i <= j+l+i_open*1.5*dk: 
                
                # note that I invert the order: l >= j >= i to follow Rustico more closely
                k1C_r.append(l)
                k2C_r.append(j)
                k3C_r.append(i)

                GetIdxB[int((l-cf)/dk),
                        int((j-cf)/dk),
                        int((i-cf)/dk)] = Ntri_tot
              
                Ntri_tot += 1

k1C_r, k2C_r, k3C_r = np.array(k1C_r), np.array(k2C_r), np.array(k3C_r)
print('Number of triangles:', Ntri_tot)

for i in range(10):
    print(k1C_r[i], k2C_r[i], k3C_r[i])

In [ ]:
#  We need the original triangle's center, in the unit of dk_ori

k1C, k2C, k3C = np.loadtxt('triangles_center.dat', unpack=True)/dk_ori

In [ ]:
# Main function

def rebin(q1E, q2E, q3E, Bisp0, numtri):

    q1E_r = np.zeros(Ntri_tot)
    q2E_r = np.zeros(Ntri_tot)
    q3E_r = np.zeros(Ntri_tot)
    Bisp0_r = np.zeros(Ntri_tot)
    numtri_r = np.zeros(Ntri_tot)

    for i in range(len(Bisp0)):
        i1 = math.floor(k1C[i]/dk - cf/dk + 1.5)
        i2 = math.floor(k2C[i]/dk - cf/dk + 1.5)
        i3 = math.floor(k3C[i]/dk - cf/dk + 1.5)

        if 1<=i1<=Nb and 1<=i2<=Nb and 1<=i3<=Nb:
            I = int(GetIdxB[int(i1-1),int(i2-1),int(i3-1)])
            
            Bisp0_r[I] += Bisp0[i]*numtri[i]
            
            q1E_r[I] += q1E[i]*numtri[i]
            q2E_r[I] += q2E[i]*numtri[i]
            q3E_r[I] += q3E[i]*numtri[i]
            
            numtri_r[I] += numtri[i]

    Bisp0_r = Bisp0_r/numtri_r
    q1E_r = q1E_r/numtri_r
    q2E_r = q2E_r/numtri_r
    q3E_r = q3E_r/numtri_r
    
    return q1E_r, q2E_r, q3E_r, Bisp0_r, numtri_r


# Rebin

In [ ]:
# The original data
fname = 'Bispectrum_data.dat'
dat = np.loadtxt(fname, unpack=True)

k1E, k2E, k3E, B0, Ntri = dat #k1 effective, k2 effective, k3effective, B0, number of fundamental triangles

# Rebinning
k1E_r, k2E_r, k3E_r, B0_r, Ntri_r = rebin(k1E, k2E, k3E, B0, Ntri)